# Comparación de Resolución Temporal y Velocidades: MATLAB vs. Veraset

Este notebook realiza una comparación estadística cuantitativa entre el dataset controlado de **MATLAB** (alta frecuencia, muestreo a 1 Hz) y el dataset de producción de **Veraset** (baja frecuencia, datos de telefonía en la vida real).

El objetivo es identificar las diferencias en:
1. **Frecuencia temporal ($\Delta t$ entre pings).**
2. **Presencia y longitud de baches (signal loss gaps).**
3. **Distribución de velocidades estimadas (calculadas homogéneamente por Haversine / $\Delta t$).**
4. **Análisis de sensibilidad del ruteador ante la pérdida de calidad de datos en 5 viajes representativos.**

Estos hallazgos servirán para calibrar las matrices probabilísticas bajo un escenario de datos degradados similar a la producción.

In [1]:
import os
from pathlib import Path
import pickle
import time
import gc
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, LineString, MultiLineString
import shapely.wkt

sns.set_theme(style="whitegrid")

## 1. Configuración de Rutas y Carga de Datos

In [2]:
# Rutas relativas del proyecto
PROJECT_ROOT = Path("../../")
GPS_DATA_DIR = PROJECT_ROOT / "Inputs" / "GPS User Data"

PATH_MATLAB = GPS_DATA_DIR / "Datos de MATLAB GPS.csv"
PATH_VERASET = GPS_DATA_DIR / "top_20users_ALL_DATES.parquet"

print("Cargando dataset de MATLAB...")
df_matlab = pd.read_csv(PATH_MATLAB, low_memory=False)
print(f"MATLAB cargado con {len(df_matlab)} filas.")

print("Cargando dataset de Veraset...")
df_veraset = pd.read_parquet(PATH_VERASET)
print(f"Veraset cargado con {len(df_veraset)} filas.")

Cargando dataset de MATLAB...
MATLAB cargado con 398043 filas.
Cargando dataset de Veraset...
Veraset cargado con 1259202 filas.


## 2. Análisis del Intervalo Temporal ($\Delta t$)

In [3]:
# Importación e ilustración de los resultados del script multi-viaje
# El análisis detallado de la física del ruteador se encuentra documentado en:
# Documentacion_Local/hallazgos_sensibilidad_ruteo.md
print("=== ESTADÍSTICAS POR MODO Y ESCENARIO (RECOPILADAS DEL EXPERIMENTO) ===")
print("\n--- VEHÍCULOS (CARRO) ---")
print("Ruta Completada - Base Raw (1Hz):     Éxito: 100.0%, Error Medio: 4.40%")
print("Ruta Completada - L1 Alta Calidad:    Éxito: 100.0%, Error Medio: 16.82%")
print("Ruta Completada - L2 Media Calidad:   Éxito: 66.7%,  Error Medio: 7.34%")
print("Ruta Completada - L3 Baja Calidad:    Éxito: 100.0%, Error Medio: 25.88%")
print("\n--- PEATONALES (CAMINAR) ---")
print("Ruta Completada - Base Raw (1Hz):     Éxito: 100.0%, Error Medio: 30.40%")
print("Ruta Completada - L1 Alta Calidad:    Éxito: 100.0%, Error Medio: 49.87%")
print("Ruta Completada - L2 Media Calidad:   Éxito: 100.0%, Error Medio: 31.90%")
print("Ruta Completada - L3 Baja Calidad:    Éxito: 100.0%, Error Medio: 38.72%")
print("\n--- ESTADÍSTICAS GLOBALES COMBINADAS ---")
print("Ruta Completada - Base Raw (1Hz):     Éxito: 100.0%, Error Medio: 14.80%")
print("Ruta Completada - L1 Alta Calidad:    Éxito: 100.0%, Error Medio: 30.04%")
print("Ruta Completada - L2 Media Calidad:   Éxito: 80.0%,  Error Medio: 19.62%")
print("Ruta Completada - L3 Baja Calidad:    Éxito: 100.0%, Error Medio: 31.02%")

=== ESTADÍSTICAS POR MODO Y ESCENARIO (RECOPILADAS DEL EXPERIMENTO) ===

--- VEHÍCULOS (CARRO) ---
Ruta Completada - Base Raw (1Hz):     Éxito: 100.0%, Error Medio: 4.40%
Ruta Completada - L1 Alta Calidad:    Éxito: 100.0%, Error Medio: 16.82%
Ruta Completada - L2 Media Calidad:   Éxito: 66.7%,  Error Medio: 7.34%
Ruta Completada - L3 Baja Calidad:    Éxito: 100.0%, Error Medio: 25.88%

--- PEATONALES (CAMINAR) ---
Ruta Completada - Base Raw (1Hz):     Éxito: 100.0%, Error Medio: 30.40%
Ruta Completada - L1 Alta Calidad:    Éxito: 100.0%, Error Medio: 49.87%
Ruta Completada - L2 Media Calidad:   Éxito: 100.0%, Error Medio: 31.90%
Ruta Completada - L3 Baja Calidad:    Éxito: 100.0%, Error Medio: 38.72%

--- ESTADÍSTICAS GLOBALES COMBINADAS ---
Ruta Completada - Base Raw (1Hz):     Éxito: 100.0%, Error Medio: 14.80%
Ruta Completada - L1 Alta Calidad:    Éxito: 100.0%, Error Medio: 30.04%
Ruta Completada - L2 Media Calidad:   Éxito: 80.0%,  Error Medio: 19.62%
Ruta Completada - L3 Baja Ca

### Gráficas de Resultados del Ruteador

#### 1. Sensibilidad Promedio del Ruteador
![Sensibilidad Ruteo](scen_1_base/sensibilidad_ruteo.png)

#### 2. Comparación Visual de Sobreposición (Overlay de Todos los Escenarios)
![Comparación Visual de Ruta](scen_1_base/comparacion_ruta_visual.png)

#### 3. Collages de Escenarios Individuales (Pings Degradados vs. Ruteo)
Haz clic en los enlaces para abrir los collages individuales y observar la ubicación de los pings degradados (marcas **X** negras):
- [Collage - Escenario Raw (1Hz)](scen_1_base/comparacion_ruta_visual_raw.png)
- [Collage - Escenario L1 (Alta Calidad - 20s)](scen_1_base/comparacion_ruta_visual_l1.png)
- [Collage - Escenario L2 (Media Calidad - 90s + Baches)](scen_1_base/comparacion_ruta_visual_l2.png)
- [Collage - Escenario L3 (Baja Calidad - 6m + Baches)](scen_1_base/comparacion_ruta_visual_l3.png)

*(Nota: Se ha exportado el archivo para Kepler.gl conteniendo los tramos de calles reconstruidos de todos los escenarios en `calibracion/rutas_kepler.csv`)*

## 3. Análisis de Baches (Signal Loss Gaps)

Evaluamos la proporción de pings que ocurren tras una desconexión de señal de corta, mediana o larga duración.

In [4]:
for threshold in [60, 300, 900]:
    gaps_m = df_matlab_clean[df_matlab_clean['delta_t'] >= threshold]
    gaps_v = df_veraset_clean[df_veraset_clean['delta_t'] >= threshold]
    
    pct_m = len(gaps_m) / len(df_matlab_clean) * 100
    pct_v = len(gaps_v) / len(df_veraset_clean) * 100
    
    print(f"Intervalos >= {threshold}s (gaps de {threshold//60} min):")
    print(f"  MATLAB: {len(gaps_m)} ({pct_m:.4f}%)")
    print(f"  VERASET: {len(gaps_v)} ({pct_v:.4f}%)\n")

Intervalos >= 60s (gaps de 1 min):
  MATLAB: 318 (0.0923%)
  VERASET: 175017 (13.8993%)

Intervalos >= 300s (gaps de 5 min):
  MATLAB: 121 (0.0351%)
  VERASET: 56193 (4.4627%)

Intervalos >= 900s (gaps de 15 min):
  MATLAB: 90 (0.0261%)
  VERASET: 17447 (1.3856%)


## 4. Análisis de la Distribución de Velocidades (Cálculo Homogéneo)

A fin de comparar magnitudes físicas equivalentes, la velocidad para ambos datasets se calcula de manera idéntica: determinando la distancia geodésica de Haversine entre posiciones sucesivas dividida entre su respectivo diferencial de tiempo $\Delta t$.

In [5]:
# Fórmula Haversine vectorizada
def haversine_np(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c
    return km

# Cálculo para MATLAB
df_matlab['lon_prev'] = df_matlab.groupby(['caid', 'num_trip'])['lon'].shift(1)
df_matlab['lat_prev'] = df_matlab.groupby(['caid', 'num_trip'])['lat'].shift(1)
df_matlab['dist_km'] = haversine_np(df_matlab['lon'], df_matlab['lat'], df_matlab['lon_prev'], df_matlab['lat_prev'])
df_matlab['speed_est'] = (df_matlab['dist_km'] / (df_matlab['delta_t'] / 3600.0))
df_matlab_speed_clean = df_matlab[(df_matlab['delta_t'] > 0) & (df_matlab['speed_est'] < 120)]

# Cálculo para VERASET
df_veraset['lon_prev'] = df_veraset.groupby('caid')['longitude'].shift(1)
df_veraset['lat_prev'] = df_veraset.groupby('caid')['latitude'].shift(1)
df_veraset['dist_km'] = haversine_np(df_veraset['longitude'], df_veraset['latitude'], df_veraset['lon_prev'], df_veraset['lat_prev'])
df_veraset['speed_est'] = (df_veraset['dist_km'] / (df_veraset['delta_t'] / 3600.0))
df_veraset_speed_clean = df_veraset[(df_veraset['delta_t'] > 0) & (df_veraset['speed_est'] < 120)]

print("=== VELOCIDADES ESTIMADAS POR HAVERSINE ===")
print("MATLAB speed_est:")
print(df_matlab_speed_clean['speed_est'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]))

print("\nVERASET speed_est:")
print(df_veraset_speed_clean['speed_est'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]))

=== VELOCIDADES ESTIMADAS POR HAVERSINE ===
MATLAB speed_est:
count    340957.000000
mean         18.767791
std          25.533256
min           0.000000
25%           0.000000
50%           0.721244
75%          36.064562
90%          53.969948
95%          72.117879
max         119.869370
Name: speed_est, dtype: float64

VERASET speed_est:
count    1.223188e+06
mean     1.620381e+01
std          25.900242
min           0.000000
25%           0.000000
50%           0.538541
75%          25.799931
90%          57.361530
95%          74.021291
max         119.999810
Name: speed_est, dtype: float64


### Gráfica de Comparación de Distribución de Velocidades Estimadas

![Distribución Velocidades](distribucion_velocidades.png)

## 5. Análisis de Sensibilidad del Ruteador a Nivel de Viaje (Multi-viaje)

Evaluamos la robustez del algoritmo de completado de rutas bajo 3 niveles de degradación de datos aplicados sobre **5 viajes representativos** (carreteras/carro y peatonales/caminar).

### Escenarios Evaluados:
* **Base Cruda (Haversine p2p):** La referencia física directa sumando la distancia Haversine entre todos los pings de 1Hz.
* **Ruta Completada - Base Raw (1Hz):** Ruteo completo del viaje utilizando todos los pings originales a 1Hz.
* **Ruta Completada - L1 Alta Calidad:** Muestreo cada 20 segundos.
* **Ruta Completada - L2 Media Calidad:** Muestreo cada 90 segundos con baches de 4 minutos.
* **Ruta Completada - L3 Baja Calidad:** Muestreo cada 6 minutos con baches severos de 12 minutos.

In [6]:
# Importación e ilustración de los resultados del script multi-viaje
# El análisis detallado de la física del ruteador se encuentra documentado en:
# Documentacion_Local/hallazgos_sensibilidad_ruteo.md
print("=== ESTADÍSTICAS GLOBALES POR ESCENARIO ===")
print("\nRuta Completada - Base Raw (1Hz):")
print("  - Tasa de Éxito en Ruteo: 5/5 (100.0%)")
print("  - Error Medio (en Ruteos Exitosos): 14.80%")
print("\nRuta Completada - L1 Alta Calidad:")
print("  - Tasa de Éxito en Ruteo: 5/5 (100.0%)")
print("  - Error Medio (en Ruteos Exitosos): 30.04%")
print("\nRuta Completada - L2 Media Calidad:")
print("  - Tasa de Éxito en Ruteo: 4/5 (80.0%)")
print("  - Error Medio (en Ruteos Exitosos): 19.62%")
print("\nRuta Completada - L3 Baja Calidad:")
print("  - Tasa de Éxito en Ruteo: 5/5 (100.0%)")
print("  - Error Medio (en Ruteos Exitosos): 31.02%")

=== ESTADÍSTICAS GLOBALES POR ESCENARIO ===

Ruta Completada - Base Raw (1Hz):
  - Tasa de Éxito en Ruteo: 5/5 (100.0%)
  - Error Medio (en Ruteos Exitosos): 14.80%

Ruta Completada - L1 Alta Calidad:
  - Tasa de Éxito en Ruteo: 5/5 (100.0%)
  - Error Medio (en Ruteos Exitosos): 30.04%

Ruta Completada - L2 Media Calidad:
  - Tasa de Éxito en Ruteo: 4/5 (80.0%)
  - Error Medio (en Ruteos Exitosos): 19.62%

Ruta Completada - L3 Baja Calidad:
  - Tasa de Éxito en Ruteo: 5/5 (100.0%)
  - Error Medio (en Ruteos Exitosos): 31.02%


### Gráfica de Sensibilidad Promedio del Ruteador

![Sensibilidad Ruteo](scen_1_base/sensibilidad_ruteo.png)

*(Nota: Se ha exportado el archivo para Kepler.gl conteniendo los tramos de calles reconstruidos de todos los escenarios en `calibracion/rutas_kepler.csv`)*